In [2]:
# Import libraries for data analysis, modeling, and plotting.
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries are ready")

Libraries are ready


In [3]:
# Load the saved Kepler dataset and preview it.
df = pd.read_csv("kepler_data.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 9564
Columns: 50


,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,...,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra_str,dec_str,koi_kepmag,koi_kepmag_err
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,4.467,0.064,-0.096,0.927,0.105,-0.061,19h27m44.22s,+48d08m29.9s,15.347,NaN
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,...,4.467,0.064,-0.096,0.927,0.105,-0.061,19h27m44.22s,+48d08m29.9s,15.347,NaN
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,0,0,0,0,...,4.544,0.044,-0.176,0.868,0.233,-0.078,19h48m01.16s,+48d08m02.9s,15.436,NaN
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,...,4.564,0.053,-0.168,0.791,0.201,-0.067,19h02m08.31s,+48d17m06.8s,15.597,NaN
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,4.438,0.070,-0.210,1.046,0.334,-0.133,19h15m01.17s,+48d13m34.3s,15.509,NaN


In [4]:
# Display the class counts and column names
print("Class counts:")
print(df["koi_disposition"].value_counts())

print("\nColumn names:")
print(df.columns.tolist())

Class counts:
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64

Column names:
['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 'koi_pdisposition', 'koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_period', 'koi_period_err1', 'koi_period_err2', 'koi_time0bk', 'koi_time0bk_err1', 'koi_time0bk_err2', 'koi_impact', 'koi_impact_err1', 'koi_impact_err2', 'koi_duration', 'koi_duration_err1', 'koi_duration_err2', 'koi_depth', 'koi_depth_err1', 'koi_depth_err2', 'koi_prad', 'koi_prad_err1', 'koi_prad_err2', 'koi_teq', 'koi_teq_err1', 'koi_teq_err2', 'koi_insol', 'koi_insol_err1', 'koi_insol_err2', 'koi_model_snr', 'koi_tce_plnt_num', 'koi_tce_delivname', 'koi_steff', 'koi_steff_err1', 'koi_steff_err2', 'koi_slogg', 'koi_slogg_err1', 'koi_slogg_err2', 'koi_srad', 'koi_srad_err1', 'koi_srad_err2', 'ra_str', 'dec_str', 'koi_kepmag', 'koi_kepmag_err']


In [5]:
# Count missing values and duplicate records.
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False).to_string())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate object names:", df["kepoi_name"].duplicated().sum())

Missing values per column:
koi_teq_err1         9564
koi_teq_err2         9564
koi_kepmag_err       9564
kepler_name          6816
koi_score            1510
koi_steff_err2        483
koi_srad_err1         468
koi_srad_err2         468
koi_slogg_err1        468
koi_steff_err1        468
koi_slogg_err2        468
koi_time0bk_err1      454
koi_period_err2       454
koi_time0bk_err2      454
koi_period_err1       454
koi_impact_err2       454
koi_depth_err2        454
koi_impact_err1       454
koi_duration_err2     454
koi_depth_err1        454
koi_duration_err1     454
koi_slogg             363
koi_srad              363
koi_steff             363
koi_prad              363
koi_depth             363
koi_prad_err1         363
koi_prad_err2         363
koi_impact            363
koi_model_snr         363
koi_teq               363
koi_tce_delivname     346
koi_tce_plnt_num      346
koi_insol_err2        321
koi_insol             321
koi_insol_err1        321
koi_kepmag              1
koi_time0bk

In [6]:
# Create a binary target for potential planets versus false positives.
data = df.copy()

data["target"] = data["koi_disposition"].map({
    "CONFIRMED": 1,
    "CANDIDATE": 1,
    "FALSE POSITIVE": 0
})

print(data["target"].value_counts().sort_index())
print("Missing targets:", data["target"].isna().sum())

target
0    4839
1    4725
Name: count, dtype: int64
Missing targets: 0


In [7]:
# Select prediction measurements and keep star IDs for splitting.
features=[
    "koi_period", "koi_impact", "koi_duration", "koi_depth",
    "koi_prad", "koi_teq", "koi_insol", "koi_model_snr",
    "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
]

X=data[features].copy()
y=data["target"].copy()
groups= data["kepid"].copy()

print("Input shape:",X.shape)
print("Missing values in selected features:")
print(X.isna().sum().to_string())

Input shape: (9564, 12)
Missing values in selected features:
koi_period         0
koi_impact       363
koi_duration       0
koi_depth        363
koi_prad         363
koi_teq          363
koi_insol        321
koi_model_snr    363
koi_steff        363
koi_slogg        363
koi_srad         363
koi_kepmag         1


In [8]:
# Split by star into training, validation, and reserved test sets.
from sklearn.model_selection import GroupShuffleSplit

test_split=GroupShuffleSplit(n_splits=1,test_size=0.20, random_state=42)
train_val_idx,test_idx= next(test_split.split(X, y, groups))
val_split=GroupShuffleSplit(n_splits=1,test_size=0.25, random_state=42)
train_part,val_part= next(
    val_split.split(X.iloc[train_val_idx],y.iloc[train_val_idx], groups.iloc[train_val_idx])
)
train_idx= train_val_idx[train_part]
val_idx =train_val_idx[val_part]
X_train,y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val= X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test= X.iloc[test_idx], y.iloc[test_idx]
groups_train= groups.iloc[train_idx]

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Validation:", X_val.shape, y_val.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())

Train: (5758, 12) {1: 2886, 0: 2872}
Validation: (1904, 12) {0: 986, 1: 918}
Test: (1902, 12) {0: 981, 1: 921}


In [9]:
# Verify that objects from one star never appear in different sets.
train_stars= set(groups.iloc[train_idx])
val_stars= set(groups.iloc[val_idx])
test_stars= set(groups.iloc[test_idx])

print("Train/validation overlap:", len(train_stars & val_stars))
print("Train/test overlap:", len(train_stars & test_stars))
print("Validation/test overlap:", len(val_stars & test_stars))

Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0


## Part 2 - Classical baseline

The logistic regression baseline is tuned with grouped cross-validation on the training set. Missing values and scaling are fitted inside each training fold.

In [10]:
# Tune logistic regression using grouped folds on the training data.
import time
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, GridSearchCV

baseline = make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
                         LogisticRegression(max_iter=2000))
parameters = {"logisticregression__C": [0.01, 0.1, 1, 10],
              "logisticregression__class_weight": [None,"balanced"]}
search=GridSearchCV(baseline, parameters, cv=GroupKFold(n_splits=3),
                    scoring="roc_auc", n_jobs=1)
start = time.perf_counter()
search.fit(X_train,y_train,groups=groups_train)
baseline_seconds=time.perf_counter()-start
baseline_model = search.best_estimator_
print("Best parameters:",search.best_params_)
print("Mean training CV ROC-AUC:",round(search.best_score_,4))
print("Tuning time (seconds):",round(baseline_seconds,2))


Best parameters: {'logisticregression__C': 10, 'logisticregression__class_weight': 'balanced'}
Mean training CV ROC-AUC: 0.8112
Tuning time (seconds): 0.44


In [11]:
# Evaluate the tuned baseline on validation data only.
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

baseline_pred=baseline_model.predict(X_val)
baseline_prob = baseline_model.predict_proba(X_val)[:,1]
baseline_results = {
    "Precision":precision_score(y_val,baseline_pred),
    "Recall": recall_score(y_val,baseline_pred),
    "F1":f1_score(y_val, baseline_pred),
    "ROC-AUC":roc_auc_score(y_val,baseline_prob)
}
print(pd.Series(baseline_results).round(4).to_string())
print("Confusion matrix [rows = actual 0,1; columns = predicted 0,1]:")
print(confusion_matrix(y_val,baseline_pred))


Precision    0.6989
Recall       0.8976
F1           0.7859
ROC-AUC      0.8061
Confusion matrix [rows = actual 0,1; columns = predicted 0,1]:
[[631 355]
 [ 94 824]]


## Part 3 - Sophisticated model

A small multilayer perceptron with two hidden layers (32 and 16 units) can represent nonlinear relationships among the same 12 measurements. This is a preliminary comparison, not a claim that it will outperform the baseline.

In [13]:
# Train a small neural network on the same training set.
from sklearn.neural_network import MLPClassifier

network=make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(32,16),alpha=0.01,
                                    max_iter=500,random_state=42))
start=time.perf_counter()
network.fit(X_train,y_train)
network_seconds = time.perf_counter() - start
print("Training time (seconds):",round(network_seconds,2))
print("Iterations:", network.named_steps["mlpclassifier"].n_iter_)


Training time (seconds): 2.08
Iterations: 291


In [14]:
# Evaluate the neural network on the shared validation set.
network_pred = network.predict(X_val)
network_prob=network.predict_proba(X_val)[:, 1]
network_results = {
    "Precision":precision_score(y_val,network_pred),
    "Recall": recall_score(y_val,network_pred),
    "F1":f1_score(y_val, network_pred),
    "ROC-AUC":roc_auc_score(y_val,network_prob)
}
print(pd.Series(network_results).round(4).to_string())
print("Confusion matrix [rows = actual 0,1; columns = predicted 0,1]:")
print(confusion_matrix(y_val,network_pred))


Precision    0.7741
Recall       0.8584
F1           0.8140
ROC-AUC      0.9015
Confusion matrix [rows = actual 0,1; columns = predicted 0,1]:
[[756 230]
 [130 788]]


In [15]:
# Compare models and inspect validation mistakes without using the test set.
comparison = pd.DataFrame([baseline_results,network_results],
                          index=["Logistic regression","Neural network"])
comparison["Training seconds"] = [baseline_seconds, network_seconds]
print(comparison.round(4).to_string())

mistakes=X_val.copy()
mistakes["actual"] = y_val
mistakes["predicted"]=network_pred
mistakes["planet_probability"] = network_prob
false_positives=mistakes[(mistakes.actual==0)&(mistakes.predicted==1)]
false_negatives=mistakes[(mistakes.actual==1)&(mistakes.predicted==0)]
print("\nFalse positives:",len(false_positives),"False negatives:",len(false_negatives))
print("\nSome false positives:")
print(false_positives[["koi_period","koi_depth","koi_model_snr",
                       "planet_probability"]].head(5).round(3).to_string())
print("\nSome false negatives:")
print(false_negatives[["koi_period","koi_depth","koi_model_snr",
                       "planet_probability"]].head(5).round(3).to_string())


                     Precision  Recall      F1  ROC-AUC  Training seconds
Logistic regression     0.6989  0.8976  0.7859   0.8061            0.4360
Neural network          0.7741  0.8584  0.8140   0.9015            2.0753

False positives: 230 False negatives: 130

Some false positives:
     koi_period  koi_depth  koi_model_snr  planet_probability
28        1.506     2211.7          181.1               0.500
384       7.047      400.9           24.0               0.630
394       2.003      260.5           22.2               0.637
569       2.867      712.4           43.0               0.766
572       2.497        NaN            NaN               0.732

Some false negatives:
     koi_period  koi_depth  koi_model_snr  planet_probability
92       19.620    21724.6          340.5               0.320
166     235.999    20714.6          271.8               0.419
212       4.408      549.4           45.1               0.097
405       7.373      205.2           19.1               0.410
498    

## Report notes and remaining plan

**Metrics.** Precision measures how often predicted potential planets are in class 1; recall measures how many class 1 objects are found. F1 combines precision and recall. ROC-AUC measures ranking across thresholds. The confusion matrix shows the counts of each error type.

**Preprocessing.** Both pipelines replace missing selected measurements with the training median and standardize from training data only. Cross-validation uses GroupKFold so the same star is never in both a training fold and its held-out fold. The test set remains untouched for the final project.

**By November 20.** Try feature ablations (such as removing stellar or transit features), tune the neural network, compare validation performance with training time and model complexity, study false positives and false negatives, and then evaluate the selected approach on the reserved test set. Interpret candidates carefully because they are not confirmed planets.


## Draft text for the November 6 progress report

### 1. Problem Statement
We use the NASA Kepler Objects of Interest cumulative table to classify an observed object as a potential planet or a false positive. The table has 9,564 objects: 2,748 CONFIRMED, 1,977 CANDIDATE, and 4,839 FALSE POSITIVE. The current binary formulation groups confirmed planets and candidates as the positive class (4,725 objects). A candidate is not necessarily a confirmed planet. Describe any change from the submitted proposal after checking its wording.

### 2. Data Preprocessing
We selected 12 transit and stellar measurements. We excluded identifiers from model inputs and excluded the disposition score, preliminary disposition, and false-positive flags because they are closely related to the target. Some selected measurements are missing; the models fill them with medians learned from training data and then standardize using training data. We split by star ID to keep planets around the same star together: 5,758 training, 1,904 validation, and 1,902 reserved test objects. There is no star overlap between splits. Preprocessing is fitted inside each training fold during baseline tuning.

### 3. Evaluation Metrics
We report precision, recall, F1, ROC-AUC, and the confusion matrix for class 1. Precision reflects how many flagged objects are potential planets, while recall reflects how many potential planets are found. F1 balances these, ROC-AUC assesses ranking across thresholds, and the confusion matrix distinguishes missed potential planets from false alarms.

### 4. Classical Baseline
Logistic regression was tuned over four regularization strengths and two class-weight settings using three-fold cross-validation grouped by star. The best setting was C = 10 with balanced class weights; mean cross-validation ROC-AUC was 0.8112. On validation, precision was 0.6989, recall 0.8976, F1 0.7859, and ROC-AUC 0.8061. The confusion matrix was [[631, 355], [94, 824]] (rows: actual 0/1; columns: predicted 0/1). Tuning took about 0.44 seconds on this run.

### 5. Sophisticated Model
A small neural network with hidden layers of 32 and 16 units was trained on the same measurements and split. It may capture nonlinear relations that logistic regression misses. On validation, precision was 0.7741, recall 0.8584, F1 0.8140, and ROC-AUC 0.9015. The confusion matrix was [[756, 230], [130, 788]]. Training took about 2.08 seconds on this run. It improved F1 and ROC-AUC but had lower recall than the baseline. These are preliminary results from one split.

### 6. Remaining Plan and Risks
Before November 20, we plan to compare feature groups (transit versus stellar), tune the neural network, compare performance with training time and model complexity, and examine more false positives and false negatives. We will use the reserved test set only after selecting an approach. Early errors include both strong and weak transit signals, so no single failure cause is clear yet; at least one false positive has missing transit measurements. Candidate labels remain uncertain because candidates are not confirmed planets. The model comparison depends on one validation split and needs further checks.
